# Cellphi example

Segments organoids with Cellpose and measures each organoid's diameter (maximum Feret diameter) in µm.

**Google Colab:** Runtime → Change runtime type → GPU, then run the install cell below.

In [ ]:
# Colab only: install Cellphi (skip this cell if you installed it locally)
# !pip install -q git+<repository URL>

In [ ]:
import cv2
import matplotlib.pyplot as plt
from cellphi import Calibration, DEFAULT_CALIBRATION, annotate, process_image, organoid_rows, write_csv
from cellphi.segment import Segmenter

segmenter = Segmenter(gpu=True)  # falls back to CPU automatically
print("Running on", segmenter.device_name)

In [ ]:
# Calibration: 1800 µm = 370 px on a 926x695 reference image. The scale is adjusted
# automatically for the image's real resolution, so nothing needs resizing.
calibration = DEFAULT_CALIBRATION
# or e.g. Calibration.from_units_per_px(2.20)

res = process_image("../input/36.jpg", segmenter, calibration)
print(f"{len(res.measured)} organoids measured, {res.units_per_px:.3f} µm/px")

In [ ]:
annotated = annotate(res.image, res.organoids, res.units_per_px)
plt.figure(figsize=(14, 10))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off");

In [ ]:
import statistics
d = res.diameters()
print(f"mean {statistics.fmean(d):.1f} µm, median {statistics.median(d):.1f} µm, sd {statistics.stdev(d):.1f} µm")
plt.hist(d, bins=20); plt.xlabel("diameter (µm)"); plt.ylabel("count");

In [ ]:
rows = organoid_rows(res.name, res.organoids, res.units_per_px, res.unit)
write_csv("36_organoids.csv", rows)
cv2.imwrite("36_annotated.jpg", annotated)